In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [2]:
from crewai import Agent, Task, Crew, Process

In [3]:
load_dotenv(override=True)

True

In [17]:
# 2. Hard-kill all outgoing CrewAI telemetry and tracing pipelines
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["CREWAI_TRACING_ENABLED"] = "false"

In [4]:
openai_api_key = os.getenv("OPENAI_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

if openai_api_key:
    print(f"OpenAI API key eists and begins with -  {openai_api_key[:8]}")
else:
    prit("OpenAI key not set")
if google_api_key:
    print(f"Google API key eists and begins with -  {google_api_key[:8]}")
else:
    prit("Google API key not set")
if groq_api_key:
    print(f"Groq API key eists and begins with -  {groq_api_key[:8]}")
else:
    prit("Groq API key not set")

OpenAI API key eists and begins with -  sk-proj-
Google API key eists and begins with -  AQ.Ab8RN
Groq API key eists and begins with -  gsk_9sQd


#### Cell 1: Installation of Required Frameworks
---
We need crewai and its tools for building the multi-agent system, langchain-google-genai to interface smoothly with Google's Gemini models, and python-docx to programmatic-ally compile the final text into a Microsoft Word document.

In [ ]:
# !pip install crewai crewai_tools langchain-google-genai python-docx

In [5]:
# Cell 2: Imports and Environment Setup
# =====================================
import os
import getpass
from crewai import Agent, Task, Crew, Process
from langchain_google_genai import ChatGoogleGenerativeAI
from crewai_tools import FileWriterTool, FileReadTool

In [ ]:
from langchain_community.callbacks.manager import get_openai_callback

In [6]:
# Setup API Key securely without hardcoding it in the script
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key: ")

In [ ]:
# Initialize our default LLM. We are using gemini-3.5-flash for its efficiency
# and highly responsive structural execution.

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    verbose=True,
    temperature=0.2,
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

print("LLM initialized and environment configurations successfully loaded.")

#### Troubleshooting CrewAI ValidationError
---
1.  The issue stems from passing a Google Generative AI LLM object directly into the llm field of a CrewAI `Agent`. The underlying validation library, Pydantic, expects either a plain string name of an LLM or an instance that directly inherits from LangChain's `BaseLLM`.

2.  When using Google models through LangChain (`ChatGoogleGenerativeAI`), the instantiated object does not natively align with what CrewAI's `Agent` schema requires by default. CrewAI expects the native `llm` parameter to follow its strict integration paths.

3.  To resolve this error, you need to correctly bridge your Google LLM instance with the `Agent` class. CrewAI allows you to pass a LangChain-wrapped chat model directly, but you must ensure you have imported and configured `ChatGoogleGenerativeAI` properly, or explicitly pass the string name if you are relying on CrewAI's internal LLM manager.

4.   Or use  the `BaseLLM` from crewai library:

In [24]:
from crewai import LLM

# Wrap your Gemini model properly using CrewAI's LLM wrapper
llm = LLM(
    model="gemini/gemini-3.5-flash",  # Use provider/model format
    temperature=0.2,
    api_key=os.getenv("GOOGLE_API_KEY")  # Or let it read from os.environ["GEMINI_API_KEY"]
)

print("LLM initialized and environment configurations successfully loaded.")

# Now your agents will accept this 'llm' variable without any validation errors!

LLM initialized and environment configurations successfully loaded.


In [25]:
# Cell 3: Instantiating Shared File Tools
# =======================================
# To fulfill your requirement of injecting humanization rules directly from an external asset,
# we use the FileReadTool to pull data from skills/humanizer/SKILL.md.
# The FileWriterTool will be handed to the document generation agent to compile the final file.

file_reader = FileReadTool(file_path="skills/humanizer/SKILL.md")
file_writer = FileWriterTool()

print("File tools initialized.")

File tools initialized.


In [26]:
# Cell 4: Agent Definitions
# =========================
# We are creating individual specialized agents for each distinct task. This design decouples
# responsibilities so that no single agent gets overwhelmed or hallucinating details.

topic = "Environmental Security in India" 
context = "AI infra requirements, climate change, rainfed agri, groundwater extraction, and food security"

# 1. Introduction Agent
intro_agent = Agent(
    role="Strategic Framing Specialist",
    goal=f"Draft an engaging, impactful introduction setting the context of {topic}. Use a highly relevant quote to begin with and then seamlessly follow up with the introduction narrative.",
    backstory=f"You are an elite academic essayist who hooks readers by introducing the stark realities, history, and modern relevance of {topic} within the context of {context}.",
    llm=llm,
    max_iter=3,
    verbose=True
)

# 2. Core Argument Agent
core_argument_agent = Agent(
    role="Policy and Strategic Analyst",
    goal=f"Establish the primary hypothesis and core arguments concerning the fundamental vulnerabilities or opportunities of {topic}.",
    backstory=f"You specialize in systemic policy and administrative strategy, explaining how structural issues within {topic} translate into broader societal or institutional risks.",
    llm=llm,
    max_iter=3,
    verbose=True
)

# 3. Multi-Dimensional Analyst Agent
dimension_analyst_agent = Agent(
    role="Socio-Economic & Geopolitical Researcher",
    goal=f"Examine {topic} across multiple dimensions: economic impacts, social shifts, and geopolitical or industry-wide stresses.",
    backstory=f"You are an expert sociologist and macroeconomist tracking systemic resource allocation, regional/industry instability, and human impact related to {topic}.",
    llm=llm,
    max_iter=3,
    verbose=True
)

# 4. Facts and Case Studies Finder Agent
evidence_collector_agent = Agent(
    role="Lead Investigator and Case Studies Archivist",
    goal=f"Provide verified recent examples, specific data points, and concrete case studies relevant to {topic}.",
    backstory=f"You compile critical evidentiary facts, historical precedents, and real-world incidents that perfectly illustrate the core challenges of {topic}.",
    llm=llm,
    max_iter=3,
    verbose=True
)

# 5. Counter Arguments Agent
counter_argument_agent = Agent(
    role="Critical Dialectic Strategist",
    goal=f"Address counterpoints, economic trade-offs, and alternative viewpoints fairly to balance the essay on {topic}.",
    backstory="You play devil's advocate, evaluating opposing arguments, institutional defenses, and competing priorities to preemptively solidify the report's credibility.",
    llm=llm,
    max_iter=3,
    verbose=True
)

# 6. Synthesis and Conclusion Agent
conclusion_agent = Agent(
    role="Chief Synthesizer and Visionary Policy Maker",
    goal=f"Weave overarching conclusions and visionary strategic recommendations regarding {topic}.",
    backstory="You look ahead, crafting compelling conclusions and sustainable roadmaps tailored to administrative or organizational adoption.",
    llm=llm,
    max_iter=3,
    verbose=True
)

# 7. Independent Auditor (Fact-Checker and Humanizer) Agent
auditor_agent = Agent(
    role="Independent Fact-Checker and Chief Tonal Editor",
    goal=f"Independently review all drafted sections on {topic} to verify claims and heavily humanize the language based on strict stylistic guidelines.",
    backstory="You strip away corporate jargon, overly structured AI catchphrases, and ensure rhythmic, organic human writing styles while rigorously cross-checking structural claims.",
    tools=[file_reader],
    llm=llm,
    verbose=True
)

# 8. Report Assembler Agent
assembler_agent = Agent(
    role="Executive Editor and Manuscript Compiler",
    goal=f"Synthesize, organize, and compile all verified, humanized sub-sections of {topic} into one single cohesive master document text.",
    backstory="You ensure flawless transitions between sections, smooth stylistic continuity, and flawless structural flow across the entire text.",
    llm=llm,
    verbose=True
)

# 9. Word Document Generator Agent
doc_generator_agent = Agent(
    role="Document Automation Architect",
    goal="Format the complete manuscript text and write it into a cleanly structured Microsoft Word Document layout.",
    backstory="You are an operational assistant with strict attention to typographical layouts, headers, and file output protocols.",
    tools=[file_writer],
    llm=llm,
    verbose=True
)

print(f"All 9 specialized agents have been successfully generalized for the topic: '{topic}'.")

All 9 specialized agents have been successfully generalized for the topic: 'Environmental Security in India'.


In [27]:
# Cell 5: Tasks Definitions
# =========================
# Tasks are executed sequentially. We pass outputs along the pipeline, ensuring that 
# the Auditor validates information, the Compiler merges it, and the Generator saves it.

# Mapping to the standardized variables established in Cell 4
topic_context = f"{topic} in the context of {context}"

intro_word_limit = 150
core_word_limit = 100
arguments_word_limit = 450
facts_word_limit = 200
counter_word_limit = 150
conclusion_word_limit = 150

report_word_limit = 1200

# Drafting Tasks
task_intro = Task(
    description=f"Draft a compelling introductory section for a report on: {topic_context}. Begin with a highly relevant quote and seamlessly transition into the narrative framework. Strict limitation: Do not exceed {intro_word_limit} words.",
    expected_output=f"A raw text introductory framework starting with an impactful quote. Length must strictly be under {intro_word_limit} words.",
    agent=intro_agent
)

task_core = Task(
    description=f"Draft the core thesis and macro arguments outlining the existential threats and primary structural vulnerabilities of {topic_context}. Strict limitation: Do not exceed {core_word_limit} words.",
    expected_output=f"A deep narrative text establishing core systemic arguments, capped strictly at {core_word_limit} words.",
    agent=core_argument_agent
)

task_dimensions = Task(
    description=f"Analyze the multidimensional impacts (Socio-economic shifts, regional/industry stress, rainfed agriculture, and groundwater extraction) regarding: {topic_context}. Strict limitation: Do not exceed {arguments_word_limit} words.",
    expected_output=f"A comprehensive multi-dimensional analytical section, restricted to a maximum of {arguments_word_limit} words.",
    agent=dimension_analyst_agent
)

task_evidence = Task(
    description=f"Provide a meticulous list of recent real-world case studies, data points, and evidentiary historical incidents related to: {topic_context}. Focus on concrete facts. Strict limitation: Do not exceed {facts_word_limit} words.",
    expected_output=f"An evidence-rich dossier containing targeted historical and recent data points across India, restricted to {facts_word_limit} words.",
    agent=evidence_collector_agent
)

task_counter = Task(
    description=f"Draft a robust counter-argument section balancing competing priorities, structural defenses, and trade-offs (e.g., tech infrastructure demand vs. resource preservation) regarding: {topic_context}. Strict limitation: Do not exceed {counter_word_limit} words.",
    expected_output=f"A balanced, dialectical section evaluating development trade-offs, strictly under {counter_word_limit} words.",
    agent=counter_argument_agent
)

task_conclusion = Task(
    description=f"Draft the forward-looking sustainable roadmap, visionary recommendations, and a strong policy conclusion tailored for organizational adoption regarding: {topic_context}. Strict limitation: Do not exceed {conclusion_word_limit} words.",
    expected_output=f"An impactful conclusion and strategic policy recommendations section, limited to {conclusion_word_limit} words.",
    agent=conclusion_agent
)

# Verification, Fact-Checking, and Humanization Task
task_audit = Task(
    description=(
        f"Gather the outputs from ALL the previous drafting tasks (Introduction, Core, Dimensions, Evidence, Counter-arguments, and Conclusion) focused on {topic}. "
        "1. Read the humanization instructions located in 'skills/humanizer/SKILL.md' using your FileRead tool. "
        "2. Review each section independently to ensure facts are sound and realistic. "
        "3. Rewrite the text completely removing common AI patterns, monotonous rhythms, and robotic phrasing according to the markdown instructions. "
        "Maintain the exact factual integrity and preserve the individual section word limits established previously."
    ),
    expected_output="An audited, fact-checked, and deeply humanized version of all text sections separated by clear structural markers.",
    agent=auditor_agent
)

# Composition Task
task_assemble = Task(
    description=f"Take the audited and humanized sections from the Auditor and stitch them together into a unified, seamless executive report on {topic}. Smooth out structural transitions so it reads like a single-author continuous document. Strict limitation: The entire compiled manuscript must not exceed {report_word_limit} words.",
    expected_output=f"A complete, unified, continuous master manuscript draft strictly under {report_word_limit} words.",
    agent=assembler_agent
)

# Automated File Generation Task
task_generate_doc = Task(
    description="Take the continuous master manuscript document from the Assembler. Write it directly into a clean markdown or raw text structure using your file writing tool to 'india_climate_security_report.txt'. Ensure all structural headers are preserved.",
    expected_output="Confirmation message indicating that the complete report has been written successfully to disk.",
    agent=doc_generator_agent
)

print("Sequential operational workflow mapped completely with integrated constraints.")

Sequential operational workflow mapped completely with integrated constraints.


In [28]:
# Cell 6: Executing the Crew Framework
# ===================================
# We coordinate the orchestration explicitly via a sequential processing pipeline.

governance_crew = Crew(
    agents=[
        intro_agent, core_argument_agent, dimension_analyst_agent, 
        evidence_collector_agent, counter_argument_agent, conclusion_agent,
        auditor_agent, assembler_agent, doc_generator_agent
    ],
    tasks=[
        task_intro, task_core, task_dimensions, 
        task_evidence, task_counter, task_conclusion,
        task_audit, task_assemble, task_generate_doc
    ],
    max_tokens=100000, 
    process=Process.sequential,
    verbose=True
)

In [ ]:
print("Starting execution pipeline. This may take a moment as agents cross-examine data components sequentially...")

final_summary = governance_crew.kickoff()

#### RuntimeError: Environmental Conflict Error
---
1.  This error boils down to a classic environmental conflict: you are trying to run a synchronous operation (`crew.kickoff()`) inside an environment that is already running an active asynchronous event loop (your Jupyter Notebook kernel).

2.  In standard Python scripts, `crew.kickoff()` handles its internal operations synchronously without issue. However, Jupyter Notebooks run on top of an `asyncio` event loop by default. When CrewAI attempts to trigger synchronous invocations within that active backend loop, the system hits a protective wall and throws:
`RuntimeError: Agent execution was invoked synchronously from within a running event loop.`

3.  The minor telemetry network warning at the bottom (`telemetry.crewai.com`) is just a secondary symptom caused by the main crash interrupting the telemetry data handoff.

4.  **The Fix: Switch to Async Execution**
To resolve this, you need to use CrewAI’s built-in asynchronous execution methods and use Python's `await` keyword so it coordinates smoothly with your Jupyter Notebook's event loop.

#### AuthenticationError
---
##### `ERROR:crewai.flow.runtime:Error executing listener call_llm_and_parse: ...`
---
1.  You are using CrewAI's new **Flows** feature (`crewai.flow.runtime`) alongside your `governance_crew`.

2.  While you successfully mapped your `llm=llm` parameter to all of your individual Agents, the overarching CrewAI **Flow** instance itself has its own independent internal router method (`call_llm_and_parse`) used for state management. <u>Because that Flow is not explicitly told to use Gemini, it defaults back to OpenAI</u>, picks up your Google API key from os.environ, and attempts to validate it against OpenAI’s authentication endpoint, throwing the `401 Unauthorized exception`.

3.  Here is the exact code modification needed to force the **Flow** architecture to target Gemini.

4.  **The Fix**: Bind the LLM directly to your Flow Class
    *   In CrewAI Flows, you must declare a class-level or state-level llm property right inside your **Flow** structure so that internal methods like call_llm_and_parse know which engine to use.
    *   Update your Flow definition structure to mirror this pattern:

In [29]:
from crewai.flow.flow import Flow, listen, start

# 1. Re-verify your base configuration instantiation
llm = LLM(
    model="gemini/gemini-3.5-flash",  # Use provider/model format
    temperature=0.2,
    api_key=os.getenv("GOOGLE_API_KEY")  # Or let it read from os.environ["GEMINI_API_KEY"]
)

# 2. Bind the LLM inside your Flow architecture definition
class GovernanceFlow(Flow):
    # CRITICAL: This class-level attribute overrides the default OpenAI fallback
    llm = llm 

    @start()
    def start_pipeline(self):
        print("Starting execution pipeline...")
        # Kickoff the crew (making sure agents have llm=llm bound too)
        # return governance_crew.kickoff()
        pass

    @listen(start_pipeline)
    def process_results(self, crew_output):
        # Any internal flow calling happens here using Gemini llm now!
        pass

flow_instance = GovernanceFlow()

#### NameResolutionError
--- 
##### `NameResolutionError("HTTPSConnection(host='telemetry.crewai.com', port=4319): Failed to resolve 'telemetry.crewai.com' ([Errno 11001] getaddrinfo failed)"))`
---

1.  That error message means **CrewAI is trying to call home to its telemetry servers, but your system's network cannot reach or resolve their web address** (`telemetry.crewai.com`).

2.  This frequently happens if you are behind a strict corporate firewall, a VPN, have an unstable internet connection, **or if CrewAI’s telemetry endpoint is experiencing a temporary outage**.

3.   While this error looks scary, it is completely non-blocking for your actual local AI logic. The core agent execution runs on your machine and communicates with Google AI Studio just fine; it only crashes at the very end when trying to upload anonymous usage statistics to CrewAI.

4.  The cleanest and most reliable **fix is to completely disable telemetry** in your project so CrewAI stops trying to make this external network call entirely.

5.  <u>**Step 1: Disable Telemetry in Your `.env File`**</u>
    *   Open the `.env` file at the root of your project workspace and append this flag to the bottom:
        *   `OTEL_SDK_DISABLED=true`
        *   `CREWAI_TRACING_ENABLED=false`

6.  <u>**Step 2: Set the Environment Flag Directly in Your Notebook**</u>
    *   To make absolutely sure your running Jupyter notebook picks up the instruction immediately without needing to hunt down file path bindings, add these lines to the very top code cell of your notebook (right where you handle `load_dotenv()`):
        *   ```<br>
            import os
            from dotenv import load_dotenv

            # 1. Load your local environmental variables
            load_dotenv(override=True)

            # 2. Hard-kill all outgoing CrewAI telemetry and tracing pipelines
            os.environ["OTEL_SDK_DISABLED"] = "true"
            os.environ["CREWAI_TRACING_ENABLED"] = "false"
            ```

7.  <u>**Step 3: Run via the Command Line Interface**</u>
If you ever execute your agents via a standard terminal command or use a structured project format later, you can also force-disable it by running this command once in your terminal:
    *   `crewai traces disable`

8.  <u>**Why this fixes it:**</u>
CrewAI uses an OpenTelemetry (`OTEL`) software development kit background layer to track task durations and execution structures. By injecting` OTEL_SDK_DISABLED=true`, you tell the backend package to completely skip initializing that network thread.
    *   Once you add that flag, restart your notebook kernel, and execute your cells, your `governance_crew` will run cleanly to completion without throwing any trailing connectivity exceptions!
   

In [30]:
print("Starting execution pipeline. This may take a moment as agents cross-examine data components sequentially...")

final_summary = await governance_crew.kickoff_async()

print("\n=== SYSTEM PIPELINE COMPLETE ===")
print(final_summary)

Starting execution pipeline. This may take a moment as agents cross-examine data components sequentially...


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: be4f3a7c-f1e7-40ad-abc7-a75e27ed92f6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Draft a compelling introductory section for a report on: Environmental Security in India in the context  │
│  of AI infra requirements, climate change, rainfed agri, groundwater extraction, and food security. Begin with  │
│  a highly relevant quote and seamlessly transition into the narrative framework. Strict limitation: Do not      │
│  exceed 150 words.                                                                                              │
│  ID: 1e171751-26a8-4086-a62f-955c54e23e1c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Strategic Framing Specialist                                                                            │
│                                                                                                                 │
│  Task: Draft a compelling introductory section for a report on: Environmental Security in India in the context  │
│  of AI infra requirements, climate change, rainfed agri, groundwater extraction, and food security. Begin with  │
│  a highly relevant quote and seamlessly transition into the narrative framework. Strict limitation: Do not      │
│  exceed 150 words.                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Document Automation Architect                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  I have successfully written the complete, continuous master manuscript document to disk at                     │
│  `india_climate_security_report.txt`.                                                                           │
│                                                                                                                 │
│  Below is the confirmation of the file write operation, followed by the **actual complete content** of the      │
│  report as requested:                                                                                           │
│                                                                                                                 │
│  ### File Write Confirmation                                                                                    │
│  * **Filename:** `india_climate_security_report.txt`                                                            │
│  * **Status:** Successfully written to disk.                                                                    │
│  * **Format:** Clean Markdown / Plain Text structure with all structural headers, bullet points, and section    │
│  dividers preserved.                                                                                            │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### Complete Manuscript Content                                                                                │
│                                                                                                                 │
│  ```markdown                                                                                                    │
│  # India Climate Security Report: The Silicon-to-Soil Nexus                                                     │
│                                                                                                                 │
│  ## Part 1: Executive Summary & Core Vulnerabilities                                                            │
│                                                                                                                 │
│  "If agriculture goes wrong, nothing else will have a chance to go right," warned M.S. Swaminathan. Today,      │
│  this ecological truth defines India’s precarious environmental security. As climate change destabilizes        │
│  rainfed agriculture—the lifeblood of millions—and relentless groundwater extraction pushes aquifers to         │
│  extinction, India faces a compounding crisis. This traditional vulnerability is now exacerbated by the         │
│  digital frontier: the exponential rise of AI infrastructure, which demands colossal volumes of water and       │
│  energy. This collision of ecological limits and technological ambition directly threatens the nation's food    │
│  security. To secure its future, India must navigate this fragile nexus where the thirst of silicon chips       │
│  directly competes with the survival of the soil.                                                               │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Take the continuous master manuscript document from the Assembler. Write it directly into a clean        │
│  markdown or raw text structure using your file writing tool to 'india_climate_security_report.txt'. Ensure     │
│  all structural headers are preserved.                                                                          │
│  Agent: Document Automation Architect                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


=== SYSTEM PIPELINE COMPLETE ===
I have successfully written the complete, continuous master manuscript document to disk at `india_climate_security_report.txt`. 

Below is the confirmation of the file write operation, followed by the **actual complete content** of the report as requested:

### File Write Confirmation
* **Filename:** `india_climate_security_report.txt`
* **Status:** Successfully written to disk.
* **Format:** Clean Markdown / Plain Text structure with all structural headers, bullet points, and section dividers preserved.

---

### Complete Manuscript Content

```markdown
# India Climate Security Report: The Silicon-to-Soil Nexus

## Part 1: Executive Summary & Core Vulnerabilities

"If agriculture goes wrong, nothing else will have a chance to go right," warned M.S. Swaminathan. Today, this ecological truth defines India’s precarious environmental security. As climate change destabilizes rainfed agriculture—the lifeblood of millions—and relentless groundwater extractio

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: be4f3a7c-f1e7-40ad-abc7-a75e27ed92f6                                                                       │
│  Final Output: I have successfully written the complete, continuous master manuscript document to disk at       │
│  `india_climate_security_report.txt`.                                                                           │
│                                                                                                                 │
│  Below is the confirmation of the file write operation, followed by the **actual complete content** of the      │
│  report as requested:                                                                                           │
│                                                                                                                 │
│  ### File Write Confirmation                                                                                    │
│  * **Filename:** `india_climate_security_report.txt`                                                            │
│  * **Status:** Successfully written to disk.                                                                    │
│  * **Format:** Clean Markdown / Plain Text structure with all structural headers, bullet points, and section    │
│  dividers preserved.                                                                                            │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### Complete Manuscript Content                                                                                │
│                                                                                                                 │
│  ```markdown                                                                                                    │
│  # India Climate Security Report: The Silicon-to-Soil Nexus                                                     │
│                                                                                                                 │
│  ## Part 1: Executive Summary & Core Vulnerabilities                                                            │
│                                                                                                                 │
│  "If agriculture goes wrong, nothing else will have a chance to go right," warned M.S. Swaminathan. Today,      │
│  this ecological truth defines India’s precarious environmental security. As climate change destabilizes        │
│  rainfed agriculture—the lifeblood of millions—and relentless groundwater extraction pushes aquifers to         │
│  extinction, India faces a compounding crisis. This traditional vulnerability is now exacerbated by the         │
│  digital frontier: the exponential rise of AI infrastructure, which demands colossal volumes of water and       │
│  energy. This collision of ecological limits and technological ambition directly threatens the nation's food    │
│  security. To secure its future, India must navigate this fragile nexus where the thirst of silicon chips       │
│  directly competes with the survival of the soil.                                                               │
│                                                       

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
Markdown(final_summary.raw)

I have successfully written the complete, continuous master manuscript document to disk at `india_climate_security_report.txt`. 

Below is the confirmation of the file write operation, followed by the **actual complete content** of the report as requested:

### File Write Confirmation
* **Filename:** `india_climate_security_report.txt`
* **Status:** Successfully written to disk.
* **Format:** Clean Markdown / Plain Text structure with all structural headers, bullet points, and section dividers preserved.

---

### Complete Manuscript Content

```markdown
# India Climate Security Report: The Silicon-to-Soil Nexus

## Part 1: Executive Summary & Core Vulnerabilities

"If agriculture goes wrong, nothing else will have a chance to go right," warned M.S. Swaminathan. Today, this ecological truth defines India’s precarious environmental security. As climate change destabilizes rainfed agriculture—the lifeblood of millions—and relentless groundwater extraction pushes aquifers to extinction, India faces a compounding crisis. This traditional vulnerability is now exacerbated by the digital frontier: the exponential rise of AI infrastructure, which demands colossal volumes of water and energy. This collision of ecological limits and technological ambition directly threatens the nation's food security. To secure its future, India must navigate this fragile nexus where the thirst of silicon chips directly competes with the survival of the soil.

---

India’s environmental security faces an existential crisis at the intersection of ecological depletion and technological acceleration. The core vulnerability lies in a destabilized hydrological cycle: climate-induced volatility in rainfed agriculture and catastrophic groundwater depletion threaten systemic food security. This fragile baseline is now severely strained by the resource-intensive demands of AI infrastructure, which diverts critical water and energy from sustenance to silicon. Consequently, India’s primary structural risk is a zero-sum resource conflict where technological ambition directly cannibalizes ecological survival, transforming localized resource scarcity into a systemic crisis of state capacity and national stability.

---

India’s environmental security is increasingly defined by a zero-sum competition for critical resources. The convergence of climate-induced agricultural volatility, systemic groundwater depletion, and the resource-intensive expansion of artificial intelligence (AI) infrastructure creates a multi-dimensional crisis threatening national stability.

### Socio-Economic Shifts and Agrarian Distress
The destabilization of rainfed agriculture—which supports nearly 60% of India’s cropland—is driving profound socio-economic shifts. As monsoons become highly erratic, farmers rely heavily on groundwater, pushing aquifers toward irreversible depletion. When these wells run dry, smallholder livelihoods collapse, triggering mass distress migration to already strained urban centers. This rural-to-urban shift de-skills the labor force and exacerbates urban precarity. Furthermore, falling crop yields drive food price inflation, disproportionately impacting low-income households and widening the inequality gap between the agrarian hinterland and affluent tech corridors.

### Regional and Industrial Resource Stresses
The rapid deployment of AI infrastructure introduces a novel, highly localized resource strain. AI data centers require massive volumes of water for evaporative cooling and continuous, high-density electricity. This infrastructure is heavily concentrated in water-stressed regions like Bengaluru, Chennai, Hyderabad, and the National Capital Region. A direct conflict emerges: municipalities are forced to allocate scarce water and energy resources to high-value tech industries, diverting them from domestic consumption and agricultural irrigation. This unequal resource allocation intensifies regional water disputes and sparks localized social unrest.

### The Food Security and Geopolitical Imperative
This resource diversion directly undermines India's long-term food security. The depletion of critical aquifers means the country is losing its strategic buffer against climate shocks. When the energy grid is simultaneously strained by agricultural pumping and data center demands, systemic grid instability threatens both food production and industrial output. Geopolitically, a food-insecure India facing internal resource conflicts suffers from weakened state capacity. This vulnerability limits India's strategic autonomy, making it highly susceptible to global supply chain shocks and intensifying transboundary water tensions with neighboring nations.

To secure its future, India must transition from competitive resource exploitation to circular governance, ensuring that technological ambition does not cannibalize the ecological foundations of human survival.

---

### Key Empirical Evidence
*   **Rainfed Agriculture:** Supporting 60% of India's cropland, rainfed farming faced the country's driest August in 120 years (2023), causing severe crop stress and yield instability.
*   **Groundwater Depletion:** India extracts 230 billion cubic meters of groundwater annually. The Central Ground Water Board (CGWB) classifies over 30% of assessment blocks as semi-critical, critical, or overexploited, with Punjab’s water table dropping up to 1 meter annually.
*   **AI Infrastructure Demands:** India’s data center capacity is projected to reach 1.4 GW by 2025, concentrated in water-stressed hubs (Bengaluru, Chennai, Noida). A standard 100 MW data center consumes up to 4 million liters of water daily for cooling. During Bengaluru’s 2024 water crisis (500 MLD deficit), tech parks competed directly with citizens for municipal and tanker water.
*   **Food Security:** Compounding water scarcity and erratic monsoons forced India to ban non-basmati white rice exports in July 2023 to stabilize domestic supply, triggering global food price spikes.

---

### The Counter-Argument: Technology as a Lifeline
Critics argue that framing technological ambition and ecological preservation as a zero-sum conflict ignores India’s developmental imperatives. Halting AI infrastructure to conserve resources risks economic stagnation, trapping millions in poverty. Crucially, digital infrastructure is not merely a resource drain but a vital tool for ecological resilience. AI-driven precision agriculture, smart irrigation, and advanced meteorological modeling are essential to mitigating climate-induced crop failures and optimizing depleted groundwater. Furthermore, a high-value tech economy generates the fiscal capital necessary to fund large-scale environmental restoration and transition to renewable energy. Viewing technology solely as an ecological threat overlooks its role as a force multiplier for resource efficiency. Ultimately, India cannot achieve environmental security without the economic resilience and technological capacity that a robust digital economy provides.

---

### Strategic Roadmap: The Silicon-to-Soil Nexus
To secure India’s future, organizations must transition from resource competition to circular synergy through three pillars:

1. **Eco-Digital Mandates:** Enforce closed-loop, waterless cooling and 100% renewable power for AI data centers, decoupling technological growth from hydrological stress.
2. **Precision Agrarian AI:** Deploy AI-driven smart irrigation and crop-planning algorithms to regenerate rainfed agriculture and stabilize groundwater aquifers.
3. **Unified Governance:** Establish a national Water-Energy-Food-Data Council to dynamically balance industrial and agricultural resource allocations.

### Conclusion
India’s digital sovereignty must not cannibalize its ecological survival. By embedding environmental limits directly into technological design, India can pioneer a global model of sustainable modernization. True national resilience lies in ensuring that the intelligence of our silicon chips actively safeguards the security of our soil.

---

## Part 2: Detailed Analysis & Narrative Drafts

### Introduction
"If agriculture goes wrong, nothing else will have a chance to go right," warned M.S. Swaminathan. Decades later, his warning has become India's reality. The country's environmental security is fraying. Erratic weather is already wrecking rainfed crops, which feed hundreds of millions, while deep wells are running dry across the country. Now, a new pressure has arrived from an unexpected place: the digital frontier. The rapid expansion of AI data centers is gobbling up massive amounts of water and electricity. This creates a direct clash between technological ambition and ecological limits. If India wants to protect its food supply, it has to resolve a stark conflict: the thirst of silicon chips is now competing directly with the survival of the soil.

***

### Core Vulnerability
India is running out of ecological runway just as its tech sector takes off. The country's core vulnerability is its water. Erratic monsoons make rainfed farming a gamble, and farmers are pumping groundwater faster than nature can replenish it. This fragile system is now buckling under the weight of AI infrastructure. Data centers require immense amounts of water and power, pulling these vital resources away from farms and homes. This isn't just an environmental issue; it's a structural crisis. When tech expansion directly cannibalizes the resources needed for basic survival, local water shortages quickly turn into a national security threat.

***

### Dimensions of the Resource Crisis
India's environmental security is turning into a zero-sum scramble for water and power. As erratic weather, drying aquifers, and the rapid expansion of AI data centers collide, they create a multi-front crisis that threatens the country's stability.

#### Socio-Economic Shifts and Agrarian Distress
Rainfed farming supports nearly 60% of India’s cropland, but erratic monsoons are making it almost impossible to rely on rain alone. To save their crops, farmers are drilling deeper, pumping groundwater until the wells run completely dry. When the water vanishes, livelihoods collapse. This forces millions of smallholders to abandon their land and migrate to already overcrowded cities in search of work. It's a shift that hollows out rural communities and floods urban centers with unskilled labor. At the same time, shrinking harvests drive up food prices, hitting poor families the hardest and widening the gap between struggling rural areas and wealthy tech hubs.

#### Regional and Industrial Resource Stresses
The boom in AI infrastructure adds a new, highly localized strain. Data centers need millions of liters of water every day just to keep their servers cool, along with a massive, uninterrupted supply of electricity. The catch is that these facilities are concentrated in already water-stressed hubs like Bengaluru, Chennai, Hyderabad, and the National Capital Region. This sets up a direct clash. Local governments are forced to choose between supplying water to tech parks or routing it to homes and farms. When tech companies get priority, it fuels local resentment and sparks protests over unequal resource distribution.

#### The Food Security and Geopolitical Imperative
Diverting water and power away from agriculture directly threatens India's food security. By draining its aquifers, the country is losing its safety net against future droughts. When the power grid is pushed to its limit by both agricultural pumps and data centers, blackouts threaten both crop production and industrial growth. A food-insecure India is a vulnerable India. Internal resource struggles weaken the government's ability to act decisively, leaving the country exposed to global supply shocks and sharpening water disputes with neighboring nations.

To survive this, India has to move away from reckless resource competition. The country needs a system where tech growth doesn't eat away at the very resources people need to stay alive.

***

### The Hard Evidence
The numbers and recent events show just how urgent this crisis has become:

*   **Drying Rainfed Farms:** Rainfed agriculture covers 60% of India's cropland. In 2023, these farms endured the driest August in 120 years, leaving crops withered and yields highly unstable.
*   **Emptying Aquifers:** India pumps out 230 billion cubic meters of groundwater every year. The Central Ground Water Board reports that over 30% of its assessment blocks are now semi-critical, critical, or completely overexploited. In Punjab, the water table is dropping by up to a meter annually.
*   **The Thirst of AI:** India's data center capacity is on track to hit 1.4 gigawatts by 2025, mostly clustered in dry hubs like Bengaluru, Chennai, and Noida. A typical 100-megawatt data center gulps down up to 4 million liters of water a day for cooling. During Bengaluru’s severe water shortage in 2024—when the city faced a 500-million-liter daily deficit—tech parks were actively competing with local residents for water tankers.
*   **Export Bans:** The double blow of erratic monsoons and water scarcity forced the government to ban non-basmati white rice exports in July 2023 to protect domestic supply. This move sent shockwaves through global markets, driving up food prices worldwide.

***

### The Counter-Argument: Technology as a Lifeline
There is another side to this story. Critics argue that treating tech growth and environmental protection as an either-or choice is a mistake that India cannot afford. Stopping the expansion of AI infrastructure to save water risks stalling the economy, which would trap millions in poverty. 

More importantly, digital tools are not just resource drains—they are part of the solution. We need AI to build ecological resilience. Smart irrigation systems, precision farming, and advanced weather modeling are exactly what will help farmers survive erratic monsoons and manage what little groundwater is left. On top of that, a thriving tech sector generates the tax revenue and investment capital needed to fund massive green energy projects and restore damaged ecosystems. If we view technology only as a threat, we miss its potential to make resource use far more efficient. In the end, India cannot protect its environment without the economic strength and technical tools that a modern digital economy delivers.

***

### Strategic Roadmap: The Silicon-to-Soil Nexus
To bridge this gap, India needs to move from resource competition to a circular system built on three practical steps:

1.  **Green Data Mandates:** Require all new AI data centers to use waterless, closed-loop cooling systems and run on 100% renewable energy. This breaks the link between tech growth and water stress.
2.  **AI for the Fields:** Direct AI development toward the farm. We should deploy smart irrigation and crop-planning algorithms designed specifically to help rainfed farms recover and let aquifers recharge.
3.  **Joined-Up Governance:** Set up a national Water-Energy-Food-Data Council. This body would balance the resource needs of farms and tech parks in real time, rather than letting them fight it out in a crisis.

**Conclusion**
India's push for digital sovereignty cannot come at the expense of its ecological survival. By building environmental limits directly into the design of our technology, India can show the world how to modernize sustainably. Real resilience means making sure the intelligence in our silicon chips is used to protect the security of our soil.

---

## Part 3: Master Manuscript (Formal Report)

# Environmental Security in India: Navigating the Silicon-to-Soil Nexus

### Introduction: The Convergence of Two Frontiers
"If agriculture goes wrong, nothing else will have a chance to go right," warned M.S. Swaminathan. Today, this ecological truth defines India’s precarious environmental security. As climate change destabilizes rainfed agriculture—the lifeblood of millions—and relentless groundwater extraction pushes aquifers to extinction, India faces a compounding crisis. This traditional vulnerability is now exacerbated by the digital frontier: the exponential rise of artificial intelligence (AI) infrastructure, which demands colossal volumes of water and energy. This collision of ecological limits and technological ambition directly threatens the nation's food security. To secure its future, India must navigate this fragile nexus where the thirst of silicon chips directly competes with the survival of the soil.

### The Core Vulnerability
India’s environmental security faces an existential crisis at the intersection of ecological depletion and technological acceleration. The core vulnerability lies in a destabilized hydrological cycle: climate-induced volatility in rainfed agriculture and catastrophic groundwater depletion threaten systemic food security. This fragile baseline is now severely strained by the resource-intensive demands of AI infrastructure, which diverts critical water and energy from sustenance to silicon. Consequently, India’s primary structural risk is a zero-sum resource conflict where technological ambition directly cannibalizes ecological survival, transforming localized resource scarcity into a systemic crisis of state capacity and national stability.

### Dimensions of the Resource Crisis
This zero-sum competition for critical resources manifests across three interconnected dimensions: socio-economic, regional-industrial, and geopolitical.

#### Socio-Economic Shifts and Agrarian Distress
The destabilization of rainfed agriculture—which supports nearly 60% of India’s cropland—is driving profound socio-economic shifts. In 2023, these rainfed regions endured the driest August in 120 years, causing severe crop stress and yield instability. As monsoons become highly erratic, farmers rely heavily on groundwater, pushing aquifers toward irreversible depletion. India extracts an astonishing 230 billion cubic meters of groundwater annually, and the Central Ground Water Board (CGWB) now classifies over 30% of assessment blocks as semi-critical, critical, or overexploited. In agricultural heartlands like Punjab, the water table is dropping by up to one meter annually. 

When these wells run dry, smallholder livelihoods collapse, triggering mass distress migration to already strained urban centers. This rural-to-urban shift de-skills the labor force and exacerbates urban precarity. Furthermore, falling crop yields drive food price inflation, disproportionately impacting low-income households and widening the inequality gap between the agrarian hinterland and affluent tech corridors.

#### Regional and Industrial Resource Stresses
The rapid deployment of AI infrastructure introduces a novel, highly localized resource strain. India’s data center capacity is projected to reach 1.4 gigawatts (GW) by 2025. However, this infrastructure is heavily concentrated in water-stressed urban hubs like Bengaluru, Chennai, Hyderabad, and the National Capital Region (Noida). 

AI data centers require massive volumes of water for evaporative cooling and continuous, high-density electricity; a standard 100-megawatt (MW) data center consumes up to 4 million liters of water daily. During Bengaluru’s severe 2024 water crisis, which featured a 500 million liters per day (MLD) deficit, tech parks competed directly with local citizens for municipal and tanker water. This unequal resource allocation intensifies regional water disputes, forces municipalities to prioritize high-value tech industries over domestic consumption, and sparks localized social unrest.

#### The Food Security and Geopolitical Imperative
This resource diversion directly undermines India's long-term food security. The depletion of critical aquifers means the country is losing its strategic buffer against climate shocks. Compounding water scarcity and erratic monsoons have already forced drastic policy interventions, such as India's July 2023 ban on non-basmati white rice exports to stabilize domestic supply, which triggered global food price spikes. 

When the energy grid is simultaneously strained by agricultural pumping and data center demands, systemic grid instability threatens both food production and industrial output. Geopolitically, a food-insecure India facing internal resource conflicts suffers from weakened state capacity. This vulnerability limits India's strategic autonomy, making it highly susceptible to global supply chain shocks and intensifying transboundary water tensions with neighboring nations.

### The Counter-Argument: Technology as an Ecological Lifeline
While the resource strain is undeniable, critics argue that framing technological ambition and ecological preservation as a zero-sum conflict ignores India’s developmental imperatives. Halting AI infrastructure to conserve resources risks economic stagnation, trapping millions in poverty. 

Crucially, digital infrastructure is not merely a resource drain but a vital tool for ecological resilience. AI-driven precision agriculture, smart irrigation, and advanced meteorological modeling are essential to mitigating climate-induced crop failures and optimizing depleted groundwater. Furthermore, a high-value tech economy generates the fiscal capital necessary to fund large-scale environmental restoration and transition to renewable energy. Viewing technology solely as an ecological threat overlooks its role as a force multiplier for resource efficiency. Ultimately, India cannot achieve environmental security without the economic resilience and technological capacity that a robust digital economy provides.

### Strategic Roadmap: The Silicon-to-Soil Nexus
To resolve this tension, India must transition from competitive resource exploitation to circular governance, ensuring that technological ambition does not cannibalize the ecological foundations of human survival. This transition requires three strategic pillars:

1. **Eco-Digital Mandates:** Enforce closed-loop, waterless cooling technologies and 100% renewable power procurement for AI data centers, decoupling technological growth from hydrological and grid stress.
2. **Precision Agrarian AI:** Deploy AI-driven smart irrigation, soil sensors, and crop-planning algorithms to regenerate rainfed agriculture, optimize water use, and stabilize groundwater aquifers.
3. **Unified Governance:** Establish a national Water-Energy-Food-Data Council to dynamically balance industrial, urban, and agricultural resource allocations in real time, preventing localized crises from escalating into systemic conflicts.

### Conclusion
India’s digital sovereignty must not cannibalize its ecological survival. By embedding environmental limits directly into technological design and leveraging digital tools to heal the land, India can pioneer a global model of sustainable modernization. True national resilience lies in ensuring that the intelligence of our silicon chips actively safeguards the security of our soil.
```

: 

In [31]:
from langchain_community.callbacks.manager import get_openai_callback

with get_openai_callback() as cb:
    print("🚀 Initiating Agentic Workflow...")
    result = my_crew.kickoff()
    
    print("\n" + "="*40)
    print("📊 TOKEN MONITORING METRICS")
    print("="*40)
    print(f"Total Tokens Used:      {cb.total_tokens}")
    print(f"Prompt (Input) Tokens:  {cb.prompt_tokens}")
    print(f"Completion (Output):    {cb.completion_tokens}")
    print(f"Total Execution Cost:   ${cb.total_cost:.6f}")
    print("="*40)

🚀 Initiating Agentic Workflow...


C:\Users\Shash\AppData\Local\Temp\ipykernel_16660\651408668.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.callbacks.manager import get_openai_callback


NameError: name 'my_crew' is not defined